# Antigen-Conditioned Antibody Developability Optimization Pipeline

Generates novel HCDR3 variants of therapeutic antibodies with better predicted
developability (HIC and AC-SINS) **while preserving antigen binding**, using
conditional flow matching in AbLang2 CDR3 embedding space with cross-attention
binding-scorer guidance.

**Architecture summary**
| Component | What it does |
|---|---|
| AbLang2 (frozen) | CDR3, framework, and full-sequence embeddings |
| Ridge oracle | Predicts HIC / AC-SINS from full-sequence embeddings |
| PLS projection | 480d CDR3 → 10d developability-aligned space for flow model |
| Flow model | Generates candidate CDR3 PLS targets per parent framework |
| **CrossAttentionBindingScorer** | **CDR3 query attends to antigen residues; pretrained on SAbDab, fine-tuned on clinical mAbs** |
| **Guided ODE** | **Binding-scorer gradient nudges ODE trajectories toward high-affinity regions at every Euler step** |
| AbLang2 masked pass | Produces natural CDR3 sequences for parent framework |
| Oracle (Phase 3) | Scores final assembled sequences |

**Run order:** top-to-bottom. Sections 2 and 4 are slow (~15 min each on T4) but cache to Drive.
Sections 5.1 and 5.3 (SAbDab pretraining) take ~30 min total and also cache.

> Runtime → Change runtime type → **T4 GPU** before running.

---
## Section 0 — Installs & Imports

In [ ]:
!pip install -q ablang2
!pip install -q torchcfm
!pip install -q torchdiffeq
!pip install -q scikit-learn scipy matplotlib seaborn joblib
!pip install -q fair-esm requests
# ANARCI: antibody sequence numbering (IMGT scheme) — used for SAbDab CDR3 extraction
!pip install -q anarci

In [ ]:
import os, re, warnings, json, contextlib, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, pearsonr, ttest_rel, ks_2samp
from scipy.spatial.distance import cdist as scipy_cdist
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
import joblib
import requests
from io import StringIO

from torchcfm.conditional_flow_matching import ConditionalFlowMatcher
import esm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/2026 Spring/BMI 702/project'
EVAL_DIR    = f'{PROJECT_DIR}/eval_inputs'
DATA_PATH   = f'{PROJECT_DIR}/GDPa1_v1.2_20250814.csv'

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR,    exist_ok=True)

CACHE = lambda name: f'{PROJECT_DIR}/{name}'

COL_VH    = 'vh_protein_sequence'
COL_VL    = 'vl_protein_sequence'
COL_HIC   = 'HIC'
COL_SINS  = 'AC-SINS_pH7.4'
COL_FOLD  = 'hierarchical_cluster_fold'
COL_AHO_H = 'heavy_aligned_aho'
COL_AHO_L = 'light_aligned_aho'
COL_NAME  = 'antibody_name'

TEST_FOLD          = 1
N_FLOW_SAMPLES     = 50
N_ODE_STEPS        = 100
M_SEQUENCES        = 20
TEMPERATURE        = 1.0
TOP_K_SELECT       = 5
FLOW_EPOCHS        = 1000
CURRENT_SIGMA      = 0.5
PLS_COMPONENTS     = 10
AA_VOCAB           = list('ACDEFGHIKLMNPQRSTVWY')
AA_TO_IDX          = {aa: i for i, aa in enumerate(AA_VOCAB)}
ABLANG_MASK        = '*'

# ── Binding-scorer hyperparameters ───────────────────────────────────────────
SCORER_HIDDEN       = 128    # shared projection dimension (CDR3 and antigen)
SCORER_HEADS        = 4      # multi-head cross-attention heads
SCORER_PRETRAIN_EP  = 100    # InfoNCE pretraining epochs on SAbDab
SCORER_FINETUNE_EP  = 50     # affinity regression fine-tuning epochs
SCORER_BATCH        = 64     # batch size for contrastive pretraining
MAX_SABDAB_ENTRIES  = 2000   # cap SAbDab fetch (increase if time permits)
ANTIGEN_MAX_LEN     = 512    # truncate antigen sequences for ESM2 (avoids OOM)
GUIDANCE_LAMBDA     = 0.5    # binding gradient scale at ODE sampling time
                              # 0 = no guidance (ablation), >1 = stronger push

---
## Section 1 — Data Loading, CDR3 Extraction & Split

### 1a — Load dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Raw rows: {df.shape[0]}')
df = df.dropna(subset=[COL_VH, COL_VL]).reset_index(drop=True)
print(f'After dropping missing VH/VL: {len(df)}')
print('\nOracle target missingness:')
print(df[[COL_HIC, COL_SINS]].isna().sum())

### 1b — HCDR3 extraction via AHo numbering

AHo CDR-H3 occupies positions 107–138 (1-indexed) = indices 106–137 (0-indexed).

In [ ]:
AHO_CDR3_START = 106
AHO_CDR3_END   = 138

def get_cdr3_and_fw_indices(aligned_aho_seq):
    cdr3_idx, fw_idx = [], []
    raw_pos = 0
    for aho_pos, char in enumerate(aligned_aho_seq):
        if char == '-':
            continue
        if AHO_CDR3_START <= aho_pos < AHO_CDR3_END:
            cdr3_idx.append(raw_pos)
        else:
            fw_idx.append(raw_pos)
        raw_pos += 1
    return cdr3_idx, fw_idx

hcdr3_seqs, h_cdr3_idx_list, h_fw_idx_list = [], [], []
for _, row in df.iterrows():
    if pd.isna(row[COL_AHO_H]):
        hcdr3_seqs.append(None); h_cdr3_idx_list.append([]); h_fw_idx_list.append([])
        continue
    cdr3_idx, fw_idx = get_cdr3_and_fw_indices(row[COL_AHO_H])
    seq = ''.join(row[COL_VH][i] for i in cdr3_idx if i < len(row[COL_VH]))
    hcdr3_seqs.append(seq if seq else None)
    h_cdr3_idx_list.append(cdr3_idx)
    h_fw_idx_list.append(fw_idx)

df['hcdr3_sequence'] = hcdr3_seqs
df['h_cdr3_idx']     = h_cdr3_idx_list
df['h_fw_idx']       = h_fw_idx_list
df['hcdr3_len']      = df['hcdr3_sequence'].apply(lambda s: len(s) if isinstance(s, str) else 0)

valid = df['hcdr3_sequence'].notna() & (df['hcdr3_sequence'] != '')
print(f'Antibodies with valid HCDR3: {valid.sum()} / {len(df)}')
print(df.loc[valid, 'hcdr3_len'].describe().round(1))

### 1c — Train/test split

In [ ]:
train_mask = df[COL_FOLD] != TEST_FOLD
test_mask  = df[COL_FOLD] == TEST_FOLD
train_idx  = df.index[train_mask].values
test_idx   = df.index[test_mask].values
print(f'Train (folds 2-5): {len(train_idx)}  |  Test (fold 1): {len(test_idx)}')

train_lens = df.loc[train_idx, 'hcdr3_len'].values
test_lens  = df.loc[test_idx,  'hcdr3_len'].values
train_lens = train_lens[train_lens > 0]
test_lens  = test_lens[test_lens > 0]

ks_stat, ks_p = ks_2samp(train_lens, test_lens)
print(f'CDR3 length KS test: stat={ks_stat:.3f}  p={ks_p:.3f}')
print('✓ compatible' if ks_p >= 0.05 else '⚠ distributions differ')

### 1d — Chemical liability annotation

In [ ]:
LIABILITY_PATTERNS = {
    'NG': re.compile(r'NG'), 'DG': re.compile(r'DG'),
    'DS': re.compile(r'DS'), 'NxS': re.compile(r'N[^P]S'),
    'NxT': re.compile(r'N[^P]T'),
}

def count_liabilities(seq):
    if not isinstance(seq, str) or not seq: return 0, {}
    counts = {n: len(p.findall(seq)) for n, p in LIABILITY_PATTERNS.items()}
    return sum(counts.values()), counts

total_liabs, liab_details = zip(*[count_liabilities(s) for s in df['hcdr3_sequence']])
df['liability_count']  = total_liabs
df['liability_detail'] = liab_details
print('Liability count distribution:')
print(df['liability_count'].value_counts().sort_index())

### 1e — HIC/AC-SINS imputation

In [ ]:
def impute_train_median(df, col, train_idx):
    vals   = df[col].copy().astype(float)
    median = float(np.nanmedian(vals.values[train_idx]))
    return vals.fillna(median).values, median

y_hic,  hic_med  = impute_train_median(df, COL_HIC,  train_idx)
y_sins, sins_med = impute_train_median(df, COL_SINS, train_idx)

y_hic_train,  y_hic_test  = y_hic[train_idx],  y_hic[test_idx]
y_sins_train, y_sins_test = y_sins[train_idx], y_sins[test_idx]

print(f'HIC  imputation median: {hic_med:.3f}')
print(f'SINS imputation median: {sins_med:.3f}')

---
## Section 1.1 — Antigen Dataset Construction, ESM2 Embeddings

Pipeline:
1. **Thera-SAbDab** → maps clinical mAb INN name to antigen target
2. **UniProt REST API** → canonical human antigen sequence
3. **ESM2-650M** → two kinds of embeddings:
   - **Mean-pooled** `(1280,)` — used for contrastive pretraining comparison
   - **Per-residue padded** `(N_ab, ANTIGEN_MAX_LEN, 1280)` — used for
     cross-attention scoring and ODE guidance

In [ ]:
print('Loading ESM2-650M...')
esm2_model, esm2_alphabet = esm.pretrained.esm2_t33_650M_UR50D()
esm2_model = esm2_model.eval().to(DEVICE)
batch_converter = esm2_alphabet.get_batch_converter()
ESM2_DIM = 1280
print('ESM2 loaded.')

In [ ]:
# ── Step 1: Thera-SAbDab lookup ──────────────────────────────────────────────
THERASABDAB_URL = (
    "https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/therasabdab/summary/"
    "?all=true&format=csv"
)
print('Downloading Thera-SAbDab...')
resp = requests.get(THERASABDAB_URL, timeout=60)
resp.raise_for_status()
thera_df = pd.read_csv(StringIO(resp.text))
thera_df['inn_lower'] = thera_df['Name'].str.lower().str.strip()
print(f'Thera-SAbDab: {len(thera_df)} entries')

ab_names = df[COL_NAME].str.lower().str.strip().tolist()
matched = []
for name in ab_names:
    row = thera_df[thera_df['inn_lower'] == name]
    target = row.iloc[0]['Target'] if len(row) > 0 else None
    matched.append({'antibody_name': name, 'thera_target': target})

antigen_df = pd.DataFrame(matched)
print(f"Matched {antigen_df['thera_target'].notna().sum()} / {len(antigen_df)} to targets")

In [ ]:
# ── Step 2: UniProt sequence fetch ───────────────────────────────────────────
def fetch_uniprot_sequence(target_name, retries=3):
    primary = re.split(r'[/;>]', target_name)[0].strip()
    base_url = "https://rest.uniprot.org/uniprotkb/search"
    for query in [
        f'gene:{primary} AND organism_id:9606 AND reviewed:true',
        f'protein_name:{primary} AND organism_id:9606 AND reviewed:true',
    ]:
        params = {"query": query, "fields": "accession,gene_names,sequence",
                  "format": "json", "size": 1}
        for attempt in range(retries):
            try:
                r = requests.get(base_url, params=params, timeout=30)
                r.raise_for_status()
                results = r.json().get("results", [])
                if results:
                    entry = results[0]
                    return {
                        "accession": entry["primaryAccession"],
                        "sequence":  entry["sequence"]["value"],
                        "gene":      entry.get("genes", [{}])[0]
                                         .get("geneName", {}).get("value", primary),
                    }
                break
            except Exception as e:
                print(f'  Retry {attempt+1} for {primary}: {e}')
                time.sleep(2 ** attempt)
    return None

print('Fetching antigen sequences from UniProt...')
sequences, accessions, genes = [], [], []
for _, row in antigen_df.iterrows():
    target = row['thera_target']
    if pd.isna(target):
        sequences.append(None); accessions.append(None); genes.append(None)
        continue
    result = fetch_uniprot_sequence(target)
    if result:
        sequences.append(result['sequence'])
        accessions.append(result['accession'])
        genes.append(result['gene'])
        print(f"  ✓ {row['antibody_name']:25s} → {target:20s} | {result['accession']} ({len(result['sequence'])} aa)")
    else:
        sequences.append(None); accessions.append(None); genes.append(None)
        print(f"  ✗ {row['antibody_name']:25s} → {target:20s} | NOT FOUND")
    time.sleep(0.3)

antigen_df['antigen_sequence']  = sequences
antigen_df['uniprot_accession'] = accessions
antigen_df['antigen_gene']      = genes
n_seq = antigen_df['antigen_sequence'].notna().sum()
print(f'\nRetrieved sequences for {n_seq} / {len(antigen_df)} antibodies')

In [ ]:
# ── Step 3: ESM2 embeddings — mean-pooled AND per-residue ────────────────────
# Both are needed:
#   mean_embs  (N, 1280) — contrastive pretraining comparison + antigen_df.csv
#   resid_embs (N, ANTIGEN_MAX_LEN, 1280) — cross-attention scoring + ODE guidance
#
# Per-residue embeddings are truncated to ANTIGEN_MAX_LEN to keep memory bounded.
# Padding is with zeros; a boolean mask tracks real vs. padded positions.

MEAN_CACHE  = f'{EVAL_DIR}/antigen_esm2_embs.npy'
RESID_CACHE = f'{EVAL_DIR}/antigen_esm2_resid.npy'
MASK_CACHE  = f'{EVAL_DIR}/antigen_valid_mask.npy'
RESID_PAD_MASK_CACHE = f'{EVAL_DIR}/antigen_resid_pad_mask.npy'

@torch.no_grad()
def embed_sequence_esm2(seq, max_len=ANTIGEN_MAX_LEN):
    seq = seq[:max_len]
    _, _, tokens = batch_converter([("antigen", seq)])
    tokens = tokens.to(DEVICE)
    out  = esm2_model(tokens, repr_layers=[33], return_contacts=False)
    reps = out["representations"][33][0, 1:-1, :]  # (seq_len, 1280)
    mean = reps.mean(dim=0).cpu().numpy()           # (1280,)
    resid = reps.cpu().numpy()                      # (seq_len, 1280)
    return mean, resid

if os.path.exists(MEAN_CACHE) and os.path.exists(RESID_CACHE):
    mean_embs    = np.load(MEAN_CACHE)
    resid_embs   = np.load(RESID_CACHE)
    valid_mask   = np.load(MASK_CACHE)
    resid_pad_mask = np.load(RESID_PAD_MASK_CACHE)
    print(f'Loaded antigen embeddings from cache.')
    print(f'  mean:  {mean_embs.shape}')
    print(f'  resid: {resid_embs.shape}')
else:
    print('Embedding antigens with ESM2 (mean-pooled + per-residue)...')
    N = len(antigen_df)
    mean_embs    = np.zeros((N, ESM2_DIM), dtype=np.float32)
    resid_embs   = np.zeros((N, ANTIGEN_MAX_LEN, ESM2_DIM), dtype=np.float32)
    resid_pad_mask = np.ones((N, ANTIGEN_MAX_LEN), dtype=bool)  # True = padding
    valid_mask   = np.zeros(N, dtype=bool)

    for i, row in antigen_df.iterrows():
        seq = row['antigen_sequence']
        if not isinstance(seq, str) or len(seq) == 0:
            continue
        try:
            mean_v, resid_v = embed_sequence_esm2(seq)
            L = min(len(resid_v), ANTIGEN_MAX_LEN)
            mean_embs[i]       = mean_v
            resid_embs[i, :L]  = resid_v[:L]
            resid_pad_mask[i, :L] = False  # real positions
            valid_mask[i]      = True
            if (i + 1) % 20 == 0:
                print(f'  {valid_mask.sum()}/{N} embedded...')
        except Exception as e:
            print(f'  ✗ {row["antibody_name"]}: {e}')

    antigen_df['esm2_valid'] = valid_mask
    antigen_df.to_csv(f'{EVAL_DIR}/antigen_df.csv', index=False)
    np.save(MEAN_CACHE,          mean_embs)
    np.save(RESID_CACHE,         resid_embs)
    np.save(MASK_CACHE,          valid_mask)
    np.save(RESID_PAD_MASK_CACHE, resid_pad_mask)
    print(f'Saved mean {mean_embs.shape}, resid {resid_embs.shape}')

# Attach to dataframe for easy downstream access
antigen_df['esm2_valid'] = valid_mask
print(f'\nAntigen coverage: {valid_mask.sum()} / {len(antigen_df)}')

---
## Section 2 — AbLang2 Embedding Extraction

| Embedding | AbLang2 mode | Used for |
|---|---|---|
| `cdr3_embs` | rescoding, mean-pool CDR3 positions | Flow model targets; scorer CDR3 input |
| `fw_embs` | rescoding with CDR3 masked to `*` | Flow model conditioning |
| `full_embs` | seqcoding (mean-pool full VH+VL) | Oracle training and scoring |

In [ ]:
import ablang2

print('Loading AbLang2 paired model (frozen)...')
ablang = ablang2.pretrained(model_to_use='ablang2-paired',
                            random_init=False, ncpu=1, device=DEVICE)
ablang.freeze()
print('AbLang2 loaded.')

### 2a — CDR3 embeddings

In [ ]:
CDR3_CACHE = CACHE('cdr3_embeddings.npy')

if os.path.exists(CDR3_CACHE):
    cdr3_embs = np.load(CDR3_CACHE)
    print(f'Loaded CDR3 embeddings: {cdr3_embs.shape}')
else:
    print('Computing CDR3 embeddings (~10 min on T4)...')
    BATCH, cdr3_list = 8, []
    for i in range(0, len(df), BATCH):
        rows = df.iloc[i:i+BATCH]
        seqs = list(zip(rows[COL_VH], rows[COL_VL]))
        with torch.no_grad():
            reps = ablang(seqs, mode='rescoding')
        for j, row in enumerate(rows.itertuples()):
            hidden = np.array(reps[j])
            vh_len = len(row.vh_protein_sequence)
            h_hid  = hidden[:vh_len]
            valid  = [k for k in row.h_cdr3_idx if k < len(h_hid)]
            cdr3_list.append(h_hid[valid].mean(0) if valid else np.zeros(480))
        if i % 40 == 0: print(f'  {min(i+BATCH, len(df))}/{len(df)}')
    cdr3_embs = np.vstack(cdr3_list)
    np.save(CDR3_CACHE, cdr3_embs)
    print(f'Saved: {cdr3_embs.shape}')

CDR3_DIM = cdr3_embs.shape[1]  # 480

### 2b — Masked framework embeddings

In [ ]:
FW_CACHE = CACHE('fw_masked_embeddings.npy')

def mask_cdr3(vh_seq, cdr3_idx):
    chars = list(vh_seq)
    for i in cdr3_idx:
        if i < len(chars): chars[i] = ABLANG_MASK
    return ''.join(chars)

if os.path.exists(FW_CACHE):
    fw_embs = np.load(FW_CACHE)
    print(f'Loaded FW embeddings: {fw_embs.shape}')
else:
    print('Computing masked-framework embeddings...')
    BATCH, fw_list = 8, []
    for i in range(0, len(df), BATCH):
        rows = df.iloc[i:i+BATCH]
        masked_seqs = [(mask_cdr3(r.vh_protein_sequence, r.h_cdr3_idx),
                        r.vl_protein_sequence) for r in rows.itertuples()]
        with torch.no_grad():
            reps = ablang(masked_seqs, mode='rescoding')
        for j, row in enumerate(rows.itertuples()):
            hidden = np.array(reps[j])
            vh_len = len(row.vh_protein_sequence)
            vl_len = len(row.vl_protein_sequence)
            n_ex   = hidden.shape[0] - vh_len - vl_len
            h_hid  = hidden[:vh_len]
            l_hid  = hidden[vh_len + n_ex:]
            fw_h   = h_hid[row.h_fw_idx].mean(0) if row.h_fw_idx else np.zeros(480)
            vl_v   = list(range(min(vl_len, l_hid.shape[0])))
            fw_l   = l_hid[vl_v].mean(0) if vl_v else np.zeros(480)
            fw_list.append((fw_h + fw_l) / 2.0)
        if i % 40 == 0: print(f'  {min(i+BATCH, len(df))}/{len(df)}')
    fw_embs = np.vstack(fw_list)
    np.save(FW_CACHE, fw_embs)
    print(f'Saved: {fw_embs.shape}')

FW_DIM = fw_embs.shape[1]

### 2c — Full-sequence embeddings (oracle input)

In [ ]:
FULL_CACHE = CACHE('full_embeddings.npy')

if os.path.exists(FULL_CACHE):
    full_embs = np.load(FULL_CACHE)
    print(f'Loaded full embeddings: {full_embs.shape}')
else:
    print('Computing full-sequence embeddings...')
    BATCH, full_list = 16, []
    for i in range(0, len(df), BATCH):
        rows = df.iloc[i:i+BATCH]
        seqs = list(zip(rows[COL_VH], rows[COL_VL]))
        with torch.no_grad():
            emb = ablang(seqs, mode='seqcoding')
        full_list.append(emb if isinstance(emb, np.ndarray) else emb.cpu().numpy())
        if i % 64 == 0: print(f'  {min(i+BATCH, len(df))}/{len(df)}')
    full_embs = np.vstack(full_list)
    np.save(FULL_CACHE, full_embs)
    print(f'Saved: {full_embs.shape}')

FULL_DIM = full_embs.shape[1]
print(f'Dims — CDR3: {CDR3_DIM}, FW: {FW_DIM}, Full: {FULL_DIM}')

### 2d — Embedding space analysis

In [ ]:
pca = PCA(n_components=50, random_state=42)
pca.fit(cdr3_embs[train_idx])
cumvar = np.cumsum(pca.explained_variance_ratio_)
n_90 = int(np.searchsorted(cumvar, 0.90)) + 1
n_50 = int(np.searchsorted(cumvar, 0.50)) + 1
print(f'CDR3 intrinsic dimensionality:')
print(f'  {n_50} PCs explain 50% variance')
print(f'  {n_90} PCs explain 90% variance')

pc_train = pca.transform(cdr3_embs[train_idx])
pc_test  = pca.transform(cdr3_embs[test_idx])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].plot(range(1, 51), cumvar * 100, lw=2, color='steelblue')
axes[0].axhline(90, color='coral', ls='--', lw=1.2, label='90%')
axes[0].axhline(50, color='gold',  ls='--', lw=1.2, label='50%')
axes[0].set(xlabel='PCs', ylabel='Cumulative variance (%)', title='CDR3 dimensionality')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
for ax, (y_tr, label, cmap) in zip(axes[1:], [
        (y_hic_train,  'HIC',     'RdYlGn_r'),
        (y_sins_train, 'AC-SINS', 'RdYlGn_r')]):
    sc = ax.scatter(pc_train[:,0], pc_train[:,1], c=y_tr, cmap=cmap, s=40, alpha=0.8)
    ax.scatter(pc_test[:,0], pc_test[:,1], c='black', marker='*', s=100, label='Test')
    plt.colorbar(sc, ax=ax, label=label)
    ax.set(xlabel='PC1', ylabel='PC2', title=f'CDR3 space — {label}')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

---
## Section 3 — Oracle Training

Ridge regression on AbLang2 full-sequence embeddings. Trained on folds 2–5 only.

In [ ]:
full_scaler = StandardScaler()
X_full_tr   = full_scaler.fit_transform(full_embs[train_idx])
X_full_te   = full_scaler.transform(full_embs[test_idx])

oracle_hic  = Ridge(alpha=0.1).fit(X_full_tr, y_hic_train)
oracle_sins = Ridge(alpha=0.1).fit(X_full_tr, y_sins_train)

pred_hic  = oracle_hic.predict(X_full_te)
pred_sins = oracle_sins.predict(X_full_te)

rho_h, _ = spearmanr(y_hic_test,  pred_hic)
rho_s, _ = spearmanr(y_sins_test, pred_sins)
r_h,   _ = pearsonr( y_hic_test,  pred_hic)
r_s,   _ = pearsonr( y_sins_test, pred_sins)

print('Oracle performance on held-out fold 1:')
print(f'  HIC  — Spearman ρ: {rho_h:.3f}   Pearson R: {r_h:.3f}')
print(f'  SINS — Spearman ρ: {rho_s:.3f}   Pearson R: {r_s:.3f}')
print('Reference (Ginkgo 2025): HIC ρ=0.42±0.09  SINS ρ=0.49±0.09')

---
## Section 4 — PLS Dimensionality Reduction for Flow Model

PLS finds directions in CDR3 embedding space that maximally co-vary with HIC and
AC-SINS. The flow model trains in 10d PLS space where every dimension is
developability-relevant.

**Critically for guidance:** PLS is a linear projection, so its approximate
inverse `x_pls → x_480 ≈ x_pls @ P^T + mean` is differentiable in PyTorch.
The binding-scorer gradient is taken w.r.t. `x_pls` and flows through this
inverse transform into the scorer — no separate architecture for 10d input needed.

In [ ]:
pls = PLSRegression(n_components=PLS_COMPONENTS)
pls.fit(
    cdr3_embs[train_idx],
    np.stack([y_hic_train, y_sins_train], axis=1))

X_cdr3_tr_pls = pls.transform(cdr3_embs[train_idx])   # (197, 10)
X_cdr3_te_pls = pls.transform(cdr3_embs[test_idx])    # (49, 10)

print(f'PLS projection: 480d → {PLS_COMPONENTS}d')
print('\nSpearman ρ between PLS components and oracle targets (training):')
for k in range(min(5, PLS_COMPONENTS)):
    rho_h, _ = spearmanr(X_cdr3_tr_pls[:, k], y_hic_train)
    rho_s, _ = spearmanr(X_cdr3_tr_pls[:, k], y_sins_train)
    print(f'  PLS{k+1}: HIC ρ={rho_h:+.3f}  SINS ρ={rho_s:+.3f}')

pls_scaler    = StandardScaler()
X_pls_tr_sc   = pls_scaler.fit_transform(X_cdr3_tr_pls)
X_pls_te_sc   = pls_scaler.transform(X_cdr3_te_pls)

fw_scaler  = StandardScaler()
cdr3_scaler = StandardScaler()
X_fw_tr    = fw_scaler.fit_transform(fw_embs[train_idx])
X_fw_te    = fw_scaler.transform(fw_embs[test_idx])
X_cdr3_tr  = cdr3_scaler.fit_transform(cdr3_embs[train_idx])
X_cdr3_te  = cdr3_scaler.transform(cdr3_embs[test_idx])

dev_scaler   = StandardScaler()
dev_train_sc = dev_scaler.fit_transform(
    np.stack([y_hic_train, y_sins_train], axis=1))

PLS_DIM = PLS_COMPONENTS
print(f'\nScalers fitted on training folds only.')

# ── PLS inverse transform (used for ODE guidance) ────────────────────────────
# PLS decomposition: X = T P^T + E, so X ≈ T P^T + mean
# pls.x_loadings_: (480, 10),  pls.x_mean_: (480,)
# In PyTorch: x_480_approx = x_pls_raw @ pls.x_loadings_.T + pls.x_mean_
# where x_pls_raw is in the *unscaled* PLS space.
#
# Since our flow model works in *scaled* PLS space (pls_scaler applied),
# the full inverse is:
#   x_pls_scaled → x_pls_raw  = x_pls_scaled * std + mean  (pls_scaler inverse)
#   x_pls_raw    → x_480_approx = x_pls_raw @ P^T + pls.x_mean_

PLS_LOADINGS_T = torch.tensor(pls.x_loadings_.T,
                               dtype=torch.float32).to(DEVICE)  # (10, 480)
PLS_MEAN       = torch.tensor(pls.x_mean_,
                               dtype=torch.float32).to(DEVICE)  # (480,)
PLS_SCALER_STD  = torch.tensor(pls_scaler.scale_,
                                dtype=torch.float32).to(DEVICE) # (10,)
PLS_SCALER_MEAN = torch.tensor(pls_scaler.mean_,
                                dtype=torch.float32).to(DEVICE) # (10,)

def inverse_pls_torch(x_pls_scaled):
    """
    Differentiable approximate inverse of the full PLS + StandardScaler pipeline.

    x_pls_scaled : (B, 10) — coordinates in scaled PLS space (flow model output)
    returns       : (B, 480) — approximate CDR3 AbLang2 embedding

    The gradient flows back through two linear operations:
      StandardScaler^{-1}: x_raw = x_scaled * std + mean
      PLS^{-1}:            x_480 = x_raw @ P^T + x_mean
    """
    x_raw  = x_pls_scaled * PLS_SCALER_STD + PLS_SCALER_MEAN  # (B, 10)
    x_480  = x_raw @ PLS_LOADINGS_T + PLS_MEAN                # (B, 480)
    return x_480

print('PLS inverse transform registered. Gradient path: x_pls_scaled → x_480.')

---
## Section 5 — Flow Model Architecture & Training

`CDR3VelocityNetPLS` is **unchanged** from the original pipeline.
The binding scorer is trained separately and only plugged in at ODE sampling time.

In [ ]:
class SinusoidalTimeEmbed(nn.Module):
    def __init__(self, dim=32):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        t    = t.view(-1, 1).float()
        half = self.dim // 2
        freqs = torch.exp(
            -np.log(10000) *
            torch.arange(half, device=t.device).float() / (half - 1))
        args = t * freqs.unsqueeze(0)
        return torch.cat([args.sin(), args.cos()], dim=-1)


class CDR3VelocityNetPLS(nn.Module):
    def __init__(self, pls_dim=10, fw_dim=480, dev_dim=2,
                 hidden=128, t_dim=32, drop=0.2):
        super().__init__()
        self.t_emb   = SinusoidalTimeEmbed(t_dim)
        self.fw_proj = nn.Sequential(
            nn.Linear(fw_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden * 2))
        self.dev_proj = nn.Sequential(
            nn.Linear(dev_dim, 32), nn.GELU(),
            nn.Linear(32, hidden))
        inp      = pls_dim + t_dim
        self.l1  = nn.Linear(inp, hidden)
        self.n1  = nn.LayerNorm(hidden)
        self.l2  = nn.Linear(hidden, hidden)
        self.n2  = nn.LayerNorm(hidden)
        self.out = nn.Linear(hidden, pls_dim)
        self.act = nn.GELU()
        self.drop = nn.Dropout(drop)

    def forward(self, t, x, fw, dev_target):
        if t.dim() == 0: t = t.expand(x.shape[0])
        t_e            = self.t_emb(t)
        h              = torch.cat([x, t_e], dim=-1)
        fw_scale, fw_b = self.fw_proj(fw).chunk(2, dim=-1)
        dev_e          = self.dev_proj(dev_target)
        h  = self.drop(self.act(self.n1(self.l1(h))))
        h  = h * (1 + fw_scale) + fw_b
        h  = h + dev_e
        h  = self.drop(self.act(self.n2(self.l2(h))))
        return self.out(h)


flow_model = CDR3VelocityNetPLS(pls_dim=PLS_DIM, fw_dim=FW_DIM).to(DEVICE)
n_params = sum(p.numel() for p in flow_model.parameters())
print(f'Flow model: {n_params:,} parameters  (working in {PLS_DIM}d PLS space)')

In [ ]:
def augment(Xp, Xf, dev, n_aug=5, noise_p=0.05, noise_f=0.01):
    ps, fs, ds = [Xp], [Xf], [dev]
    for _ in range(n_aug):
        ps.append(Xp + np.random.normal(0, noise_p, Xp.shape))
        fs.append(Xf + np.random.normal(0, noise_f, Xf.shape))
        ds.append(dev)
    return np.vstack(ps), np.vstack(fs), np.vstack(ds)

Xp_aug, Xf_aug, dev_aug = augment(X_pls_tr_sc, X_fw_tr, dev_train_sc, n_aug=5)
ds = TensorDataset(
    torch.tensor(Xp_aug,  dtype=torch.float32),
    torch.tensor(Xf_aug,  dtype=torch.float32),
    torch.tensor(dev_aug, dtype=torch.float32))
loader = DataLoader(ds, batch_size=32, shuffle=True)
print(f'Augmented training set: {Xp_aug.shape}')

In [ ]:
FLOW_CKPT = CACHE(f'flow_model_pls_sigma{CURRENT_SIGMA}.pt')
FM  = ConditionalFlowMatcher(sigma=CURRENT_SIGMA)
opt = AdamW(flow_model.parameters(), lr=1e-3, weight_decay=1e-4)
sch = CosineAnnealingLR(opt, T_max=FLOW_EPOCHS, eta_min=1e-5)
losses = []

print(f'Training flow model in {PLS_DIM}d PLS space (sigma={CURRENT_SIGMA})...')
for epoch in range(FLOW_EPOCHS):
    flow_model.train()
    ep = 0.0
    for x1, fw, dev in loader:
        x1, fw, dev = x1.to(DEVICE), fw.to(DEVICE), dev.to(DEVICE)
        x0 = torch.randn_like(x1)
        t, xt, ut = FM.sample_location_and_conditional_flow(x0, x1)
        vt = flow_model(t, xt, fw, dev)
        loss = F.mse_loss(vt, ut)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(flow_model.parameters(), 1.0)
        opt.step(); ep += loss.item()
    sch.step(); losses.append(ep / len(loader))
    if (epoch + 1) % 200 == 0:
        print(f'  Epoch {epoch+1:4d} | CFM loss: {losses[-1]:.5f}')

torch.save(flow_model.state_dict(), FLOW_CKPT)
np.save(CACHE('flow_losses.npy'), np.array(losses))
print(f'Saved → {FLOW_CKPT}')

In [ ]:
# ── Load checkpoint (skip training if already done) ──────────────────────────
flow_model = CDR3VelocityNetPLS(pls_dim=PLS_DIM, fw_dim=FW_DIM).to(DEVICE)
flow_model.load_state_dict(torch.load(FLOW_CKPT, map_location=DEVICE))
flow_model.eval()
losses = np.load(CACHE('flow_losses.npy')).tolist()

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, lw=1.2, color='steelblue')
ax.set(xlabel='Epoch', ylabel='CFM MSE loss', yscale='log',
       title='Flow model training loss')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f'Flow model loaded.')

---
## Section 5.1 — SAbDab Pretraining Data Pipeline

Collect (HCDR3 CDR3 embedding, antigen mean-pooled ESM2 embedding) pairs from
SAbDab for contrastive pretraining.  Clinical mAbs already in the Ginkgo dataset
are excluded so the scorer never trains on its own test distribution.

**Steps**
1. Download SAbDab summary → filter for protein antigens with both H+L chains
2. Batch-fetch H chain + antigen chain sequences from RCSB GraphQL API
3. Extract HCDR3 with ANARCI (IMGT scheme, positions 105–117)
4. Compute AbLang2 CDR3 embeddings → project through existing PLS
5. Compute ESM2 mean-pooled antigen embeddings
6. Collect affinity labels (`affinity_nM` column) for fine-tuning step

In [ ]:
from anarci import anarci as anarci_number

# ── Step 1: Download SAbDab summary ──────────────────────────────────────────
SABDAB_URL  = ("https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab/"
               "summary/?all=true&format=csv")
SABDAB_CACHE = CACHE('sabdab_summary.csv')

if os.path.exists(SABDAB_CACHE):
    sabdab_raw = pd.read_csv(SABDAB_CACHE)
    print(f'Loaded SAbDab summary from cache: {len(sabdab_raw)} entries')
else:
    print('Downloading SAbDab summary (~5 MB)...')
    resp = requests.get(SABDAB_URL, timeout=120)
    resp.raise_for_status()
    sabdab_raw = pd.read_csv(StringIO(resp.text))
    sabdab_raw.to_csv(SABDAB_CACHE, index=False)
    print(f'Saved SAbDab summary: {len(sabdab_raw)} entries')

print(f'SAbDab columns: {list(sabdab_raw.columns)}')

In [ ]:
# ── Step 2: Filter SAbDab ─────────────────────────────────────────────────────
# Keep: protein antigens, both H and L chains present, PDB format
# Exclude: entries from our clinical mAb dataset

# Normalise column names (SAbDab column names vary slightly across versions)
col_map = {}
for col in sabdab_raw.columns:
    lc = col.lower().replace(' ', '_')
    if 'hchain' in lc or lc == 'hchain':          col_map[col] = 'Hchain'
    elif 'lchain' in lc or lc == 'lchain':        col_map[col] = 'Lchain'
    elif 'antigen_chain' in lc:                   col_map[col] = 'antigen_chain'
    elif 'antigen_type' in lc:                    col_map[col] = 'antigen_type'
    elif lc == 'pdb':                             col_map[col] = 'pdb'
    elif 'affinity' in lc and 'method' not in lc: col_map[col] = 'affinity_nM'
    elif 'delta_g' in lc:                         col_map[col] = 'delta_g'
sabdab_raw = sabdab_raw.rename(columns=col_map)

required = ['pdb', 'Hchain', 'Lchain', 'antigen_chain']
missing  = [c for c in required if c not in sabdab_raw.columns]
if missing:
    raise RuntimeError(f'SAbDab summary missing columns: {missing}. '
                       f'Available: {list(sabdab_raw.columns)}')

sabdab = sabdab_raw.copy()

# Protein antigen filter
if 'antigen_type' in sabdab.columns:
    sabdab = sabdab[sabdab['antigen_type'].str.contains('protein', case=False, na=False)]

# Must have H + L chains and antigen chain
sabdab = sabdab[
    sabdab['Hchain'].notna() &
    sabdab['Lchain'].notna() &
    sabdab['antigen_chain'].notna()
]

# Exclude clinical mAbs already in the Ginkgo dataset
# (identified via Thera-SAbDab PDB cross-reference)
clinical_pdbs = set()
if 'pdb' in thera_df.columns:
    clinical_pdbs = set(thera_df['pdb'].dropna().str.lower())
    sabdab = sabdab[~sabdab['pdb'].str.lower().isin(clinical_pdbs)]

# Cap for speed (increase MAX_SABDAB_ENTRIES if you have time)
sabdab = sabdab.sample(min(MAX_SABDAB_ENTRIES, len(sabdab)),
                        random_state=42).reset_index(drop=True)

print(f'SAbDab entries after filtering: {len(sabdab)}')
if 'affinity_nM' in sabdab.columns:
    n_aff = sabdab['affinity_nM'].notna().sum()
    print(f'  Entries with affinity labels: {n_aff}')

In [ ]:
# ── Step 3: Batch-fetch chain sequences from RCSB GraphQL ─────────────────────
RCSB_GRAPHQL = "https://data.rcsb.org/graphql"

def fetch_rcsb_sequences_batch(pdb_ids):
    """
    Fetch all polymer chain sequences for a list of PDB IDs.
    Returns dict {pdb_id: {chain_id: sequence}}.
    """
    ids_json = json.dumps([p.upper() for p in pdb_ids])
    query = f"""
    {{
      entries(entry_ids: {ids_json}) {{
        rcsb_id
        polymer_entities {{
          entity_poly {{
            pdbx_seq_one_letter_code
            pdbx_strand_id
          }}
        }}
      }}
    }}
    """
    try:
        r = requests.post(RCSB_GRAPHQL, json={"query": query}, timeout=60)
        r.raise_for_status()
        entries = r.json().get("data", {}).get("entries", []) or []
    except Exception as e:
        print(f'  RCSB batch error: {e}')
        return {}

    result = {}
    for entry in entries:
        if entry is None: continue
        pdb = entry["rcsb_id"].lower()
        chain_seqs = {}
        for entity in (entry.get("polymer_entities") or []):
            poly = entity.get("entity_poly") or {}
            seq  = poly.get("pdbx_seq_one_letter_code", "")
            # pdbx_strand_id is comma-separated chain IDs that share this entity
            for ch in (poly.get("pdbx_strand_id") or "").split(","):
                chain_seqs[ch.strip()] = seq
        result[pdb] = chain_seqs
    return result

SABDAB_SEQ_CACHE = CACHE('sabdab_sequences.json')

if os.path.exists(SABDAB_SEQ_CACHE):
    with open(SABDAB_SEQ_CACHE) as f:
        sabdab_seqs = json.load(f)
    print(f'Loaded RCSB sequences from cache: {len(sabdab_seqs)} PDB entries')
else:
    print('Fetching sequences from RCSB GraphQL (batched)...')
    sabdab_seqs = {}
    all_pdbs = sabdab['pdb'].str.lower().unique().tolist()
    BATCH_SIZE = 50
    for start in range(0, len(all_pdbs), BATCH_SIZE):
        batch = all_pdbs[start:start + BATCH_SIZE]
        batch_result = fetch_rcsb_sequences_batch(batch)
        sabdab_seqs.update(batch_result)
        print(f'  {min(start + BATCH_SIZE, len(all_pdbs))}/{len(all_pdbs)} PDB entries fetched')
        time.sleep(0.2)
    with open(SABDAB_SEQ_CACHE, 'w') as f:
        json.dump(sabdab_seqs, f)
    print(f'Saved {len(sabdab_seqs)} PDB sequence records')

In [ ]:
# ── Step 4: Extract HCDR3 using ANARCI (IMGT numbering) ──────────────────────
# IMGT CDR-H3: positions 105–117 (inclusive).
# anarci returns: [[(numbered_seq, chain_type, species)], [hit_info]] per query
# numbered_seq is a list of ((imgt_pos, insert_code), aa_char) tuples.

IMGT_CDR3_START = 105
IMGT_CDR3_END   = 117

def extract_hcdr3_imgt(vh_seq):
    """Return HCDR3 string (may be empty) or None if numbering failed."""
    if not isinstance(vh_seq, str) or len(vh_seq) < 30:
        return None
    try:
        results = anarci_number([(0, vh_seq)], scheme='imgt', output=False)
        if not results or results[0] is None or results[0][0] is None:
            return None
        numbered = results[0][0][0]  # list of ((pos, ins), aa) tuples
        cdr3 = ''.join(
            aa for (pos, _), aa in numbered
            if IMGT_CDR3_START <= pos <= IMGT_CDR3_END and aa != '-'
        )
        return cdr3 if cdr3 else None
    except Exception:
        return None

print('Extracting HCDR3 sequences with ANARCI...')
vh_seqs, vl_seqs, ag_seqs, hcdr3_seqs_sab, affinities = [], [], [], [], []
pdb_ids_valid = []

for _, row in sabdab.iterrows():
    pdb = row['pdb'].lower()
    chain_map = sabdab_seqs.get(pdb, {})
    if not chain_map:
        continue

    # Heavy chain — first listed chain ID
    h_chain = str(row['Hchain']).split('|')[0].strip()
    vh = chain_map.get(h_chain) or chain_map.get(h_chain.upper())
    if not vh:
        continue

    # Light chain
    l_chain = str(row['Lchain']).split('|')[0].strip()
    vl = chain_map.get(l_chain) or chain_map.get(l_chain.upper())
    if not vl:
        continue

    # Antigen chain (take first if multiple listed)
    ag_chain = str(row['antigen_chain']).split('|')[0].split(',')[0].strip()
    ag = chain_map.get(ag_chain) or chain_map.get(ag_chain.upper())
    if not ag or len(ag) < 20:   # skip tiny antigen fragments
        continue

    # HCDR3 via IMGT
    hcdr3 = extract_hcdr3_imgt(vh)
    if not hcdr3 or len(hcdr3) < 3:
        continue

    vh_seqs.append(vh)
    vl_seqs.append(vl)
    ag_seqs.append(ag)
    hcdr3_seqs_sab.append(hcdr3)
    pdb_ids_valid.append(pdb)

    aff = None
    if 'affinity_nM' in row and pd.notna(row['affinity_nM']):
        try: aff = float(row['affinity_nM'])
        except: pass
    affinities.append(aff)

sabdab_valid = pd.DataFrame({
    'pdb':       pdb_ids_valid,
    'vh':        vh_seqs,
    'vl':        vl_seqs,
    'ag':        ag_seqs,
    'hcdr3':     hcdr3_seqs_sab,
    'affinity_nM': affinities,
})

print(f'Valid SAbDab entries: {len(sabdab_valid)}')
print(f'  With affinity labels: {sabdab_valid["affinity_nM"].notna().sum()}')
print(f'  HCDR3 length: mean={np.mean([len(s) for s in hcdr3_seqs_sab]):.1f}')

In [ ]:
# ── Step 5: AbLang2 CDR3 embeddings for SAbDab entries ──────────────────────
# We embed the full (VH, VL) pair with AbLang2 rescoding, then mean-pool the
# CDR3 positions (identified from IMGT extraction above) to get a 480d vector.
# The IMGT CDR3 positions are re-derived from AHo-equivalent positions using
# a simple ANARCI alignment step for the raw VH sequence.

SAB_CDR3_EMB_CACHE = CACHE('sabdab_cdr3_embs.npy')
SAB_PLS_EMB_CACHE  = CACHE('sabdab_pls_embs.npy')

def get_hcdr3_positions_from_seq(vh_seq, hcdr3_seq):
    """
    Locate CDR3 positions in the raw VH sequence by finding the longest
    exact match of the extracted HCDR3 string.  Falls back to the last
    occurrence of the first 4 and last 4 residues as anchors.
    Returns list of integer indices into vh_seq, or None.
    """
    idx = vh_seq.find(hcdr3_seq)
    if idx >= 0:
        return list(range(idx, idx + len(hcdr3_seq)))
    # Partial anchor fallback
    anchor = hcdr3_seq[:4]
    idx2 = vh_seq.rfind(anchor)
    if idx2 >= 0:
        end = min(idx2 + len(hcdr3_seq), len(vh_seq))
        return list(range(idx2, end))
    return None

if os.path.exists(SAB_CDR3_EMB_CACHE):
    sab_cdr3_embs = np.load(SAB_CDR3_EMB_CACHE)
    sab_pls_embs  = np.load(SAB_PLS_EMB_CACHE)
    print(f'Loaded SAbDab CDR3 embs: {sab_cdr3_embs.shape}')
    print(f'Loaded SAbDab PLS embs:  {sab_pls_embs.shape}')
else:
    print('Computing AbLang2 CDR3 embeddings for SAbDab (~20 min)...')
    BATCH = 8
    N = len(sabdab_valid)
    sab_cdr3_list, keep_mask = [], np.ones(N, dtype=bool)

    for i in range(0, N, BATCH):
        batch_rows = sabdab_valid.iloc[i:i+BATCH]
        seqs = list(zip(batch_rows['vh'], batch_rows['vl']))
        try:
            with torch.no_grad():
                reps = ablang(seqs, mode='rescoding')
        except Exception as e:
            print(f'  AbLang2 error at batch {i}: {e}')
            for j in range(len(batch_rows)):
                sab_cdr3_list.append(np.zeros(480))
                keep_mask[i + j] = False
            continue

        for j, row in enumerate(batch_rows.itertuples()):
            hidden = np.array(reps[j])
            vh_len = len(row.vh)
            h_hid  = hidden[:vh_len]
            pos    = get_hcdr3_positions_from_seq(row.vh, row.hcdr3)
            if pos:
                valid = [k for k in pos if k < len(h_hid)]
                emb   = h_hid[valid].mean(0) if valid else np.zeros(480)
            else:
                emb = np.zeros(480)
                keep_mask[i + j] = False
            sab_cdr3_list.append(emb)

        if i % 80 == 0:
            print(f'  {min(i+BATCH, N)}/{N} processed')

    sab_cdr3_embs = np.vstack(sab_cdr3_list)

    # Project through the PLS fitted on clinical mAbs
    # (PLS directions encode developability; same projections for SAbDab)
    sab_pls_raw   = pls.transform(sab_cdr3_embs)   # (N, 10)
    sab_pls_embs  = pls_scaler.transform(sab_pls_raw)

    # Remove entries where embedding extraction failed
    sabdab_valid['keep'] = keep_mask
    sabdab_valid = sabdab_valid[keep_mask].reset_index(drop=True)
    sab_cdr3_embs = sab_cdr3_embs[keep_mask]
    sab_pls_embs  = sab_pls_embs[keep_mask]

    np.save(SAB_CDR3_EMB_CACHE, sab_cdr3_embs)
    np.save(SAB_PLS_EMB_CACHE,  sab_pls_embs)
    print(f'Saved CDR3 embs: {sab_cdr3_embs.shape}')

In [ ]:
# ── Step 6: ESM2 mean-pooled antigen embeddings for SAbDab ───────────────────
# Only mean-pooled (not per-residue) is needed for contrastive pretraining.
# Per-residue embeddings for the clinical mAbs are already in resid_embs.

SAB_AG_EMB_CACHE = CACHE('sabdab_ag_embs.npy')

if os.path.exists(SAB_AG_EMB_CACHE):
    sab_ag_embs = np.load(SAB_AG_EMB_CACHE)
    print(f'Loaded SAbDab antigen embs: {sab_ag_embs.shape}')
else:
    print('Embedding SAbDab antigens with ESM2 (mean-pooled)...')
    N = len(sabdab_valid)
    sab_ag_embs = np.zeros((N, ESM2_DIM), dtype=np.float32)
    for i, row in sabdab_valid.iterrows():
        try:
            mean_v, _ = embed_sequence_esm2(row['ag'])
            sab_ag_embs[i] = mean_v
        except Exception as e:
            print(f'  ✗ row {i}: {e}')
        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{N} embedded')
    np.save(SAB_AG_EMB_CACHE, sab_ag_embs)
    print(f'Saved SAbDab antigen embs: {sab_ag_embs.shape}')

print(f'\nSAbDab pretraining set: {len(sabdab_valid)} (CDR3, antigen) pairs')
n_with_aff = sabdab_valid['affinity_nM'].notna().sum()
print(f'  With affinity labels for fine-tuning: {n_with_aff}')

---
## Section 5.2 — CrossAttentionBindingScorer Architecture

```
CrossAttentionBindingScorer
├── cdr3_proj   Linear(480, 128) + LayerNorm    # shared: trained in both stages
├── ag_proj     Linear(1280, 128) + LayerNorm   # shared: trained in both stages
├── cross_attn  MultiheadAttention(128, 4 heads)
├── score_head  MLP(256→128→1)
└── log_temp    learnable scalar (contrastive temperature)
```

**Two forward modes share `cdr3_proj` and `ag_proj`:**

| Mode | Input antigen | Used for |
|---|---|---|
| `forward_contrastive` | mean-pooled `(B, 1280)` | Stage 1 InfoNCE pretraining on SAbDab |
| `forward_score` | per-residue `(B, L, 1280)` | Stage 2 fine-tuning + ODE guidance gradient |

**Why share projections across modes?**
Contrastive pretraining with thousands of SAbDab pairs forces `cdr3_proj` and
`ag_proj` to learn representations where bound pairs are geometrically close.
Stage 2 fine-tuning inherits this initialization for the cross-attention layers,
which otherwise have only a handful of labeled pairs to learn from.

In [ ]:
class CrossAttentionBindingScorer(nn.Module):
    """
    Dual-mode binding scorer for antibody–antigen affinity estimation.

    Parameters
    ----------
    cdr3_dim : int  — dimensionality of AbLang2 CDR3 embedding (480)
    ag_dim   : int  — dimensionality of ESM2 antigen embedding (1280)
    hidden   : int  — shared projection space (128)
    n_heads  : int  — multi-head attention heads (4)
    dropout  : float
    """

    def __init__(self, cdr3_dim=480, ag_dim=ESM2_DIM,
                 hidden=SCORER_HIDDEN, n_heads=SCORER_HEADS, dropout=0.1):
        super().__init__()

        # ── Shared projections (trained in both Stage 1 and Stage 2) ─────────
        self.cdr3_proj = nn.Sequential(
            nn.Linear(cdr3_dim, hidden, bias=False),
            nn.LayerNorm(hidden),
        )
        self.ag_proj = nn.Sequential(
            nn.Linear(ag_dim, hidden, bias=False),
            nn.LayerNorm(hidden),
        )

        # ── Cross-attention: CDR3 query attends to antigen residues ──────────
        # batch_first=True: input/output shape is (B, seq, dim)
        self.cross_attn = nn.MultiheadAttention(
            hidden, n_heads, dropout=dropout, batch_first=True)

        # ── Scoring head (Stage 2 + guidance) ────────────────────────────────
        # Concatenates CDR3 projection + attention context before predicting
        self.score_head = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

        # Learnable log-temperature for InfoNCE loss (initialised to log(0.07))
        self.log_temp = nn.Parameter(torch.tensor(-2.66))

    # ── Mode 1: dual-encoder contrastive (Stage 1) ───────────────────────────
    def forward_contrastive(self, cdr3_emb, ag_emb_mean):
        """
        Project both modalities to 128d and L2-normalise.

        cdr3_emb    : (B, 480) — AbLang2 mean-pooled CDR3 embedding
        ag_emb_mean : (B, 1280) — ESM2 mean-pooled antigen embedding

        Returns
        -------
        z_ab : (B, 128) L2-normalised antibody embeddings
        z_ag : (B, 128) L2-normalised antigen embeddings
        """
        z_ab = F.normalize(self.cdr3_proj(cdr3_emb),    dim=-1)
        z_ag = F.normalize(self.ag_proj(ag_emb_mean),   dim=-1)
        return z_ab, z_ag

    # ── Mode 2: cross-attention scorer (Stage 2 + ODE guidance) ─────────────
    def forward_score(self, cdr3_emb, ag_residues, ag_key_padding_mask=None):
        """
        CDR3 embedding (query) attends to antigen residue embeddings (keys/values).

        cdr3_emb          : (B, 480) — AbLang2 CDR3 embedding
        ag_residues        : (B, L, 1280) — per-residue ESM2 embeddings
        ag_key_padding_mask: (B, L) bool, True = padding position (ignored)

        Returns
        -------
        score : (B,) — predicted binding score (higher = stronger binding)
        """
        # Project CDR3 to query: (B, 1, hidden)
        q = self.cdr3_proj(cdr3_emb).unsqueeze(1)

        # Project antigen residues to keys/values: (B, L, hidden)
        kv = self.ag_proj(ag_residues)

        # Cross-attention — CDR3 reads the antigen
        context, attn_weights = self.cross_attn(
            q, kv, kv,
            key_padding_mask=ag_key_padding_mask)  # (B, 1, hidden)

        # Combine CDR3 representation with antigen-conditioned context
        combined = torch.cat([
            q.squeeze(1),        # (B, hidden)
            context.squeeze(1),  # (B, hidden)
        ], dim=-1)               # (B, 2*hidden)

        return self.score_head(combined).squeeze(-1)  # (B,)

    @property
    def temperature(self):
        # Clamp to [0.01, 1.0] for numerical stability
        return self.log_temp.exp().clamp(0.01, 1.0)


binding_scorer = CrossAttentionBindingScorer().to(DEVICE)
n_scorer_params = sum(p.numel() for p in binding_scorer.parameters())
print(f'CrossAttentionBindingScorer: {n_scorer_params:,} parameters')
print(f'  cdr3_proj  : (480) → (128)')
print(f'  ag_proj    : (1280) → (128)')
print(f'  cross_attn : MultiheadAttention(128, {SCORER_HEADS} heads)')
print(f'  score_head : MLP(256 → 128 → 1)')

---
## Section 5.3 — Stage 1: Contrastive Pretraining on SAbDab

**Loss:** Symmetric InfoNCE (NT-Xent) with in-batch negatives.

For a batch of size B, the antibody and antigen embeddings form a (B×B) similarity
matrix. The diagonal entries are positive (matched) pairs; all off-diagonal entries
are negatives. Cross-entropy is applied twice — once treating each antibody as a
query over antigens, once the reverse — and averaged.

**What this trains:** `cdr3_proj` and `ag_proj` are pushed to map bound
pairs close together in 128d space, giving a strong initialisation for Stage 2.

In [ ]:
def infonce_loss(z_ab, z_ag, temperature):
    """
    Symmetric InfoNCE loss.

    z_ab, z_ag : (B, 128) L2-normalised embeddings
    temperature : scalar > 0

    Returns scalar loss (lower = better alignment of matched pairs).
    """
    B   = z_ab.shape[0]
    sim = (z_ab @ z_ag.T) / temperature    # (B, B)
    labels = torch.arange(B, device=z_ab.device)
    loss_ab = F.cross_entropy(sim,   labels)   # antibody → antigen
    loss_ag = F.cross_entropy(sim.T, labels)   # antigen → antibody
    return (loss_ab + loss_ag) / 2.0


SCORER_PRETRAIN_CKPT = CACHE('binding_scorer_pretrained.pt')

if os.path.exists(SCORER_PRETRAIN_CKPT):
    binding_scorer.load_state_dict(
        torch.load(SCORER_PRETRAIN_CKPT, map_location=DEVICE))
    binding_scorer.eval()
    print(f'Loaded pretrained scorer from {SCORER_PRETRAIN_CKPT}')
else:
    print(f'Stage 1 — contrastive pretraining on {len(sabdab_valid)} SAbDab pairs')
    print(f'  Epochs: {SCORER_PRETRAIN_EP}  |  Batch: {SCORER_BATCH}\n')

    # Build dataset: (CDR3 emb 480d, antigen mean emb 1280d)
    pretrain_ds = TensorDataset(
        torch.tensor(sab_cdr3_embs, dtype=torch.float32),
        torch.tensor(sab_ag_embs,   dtype=torch.float32),
    )
    pretrain_loader = DataLoader(
        pretrain_ds, batch_size=SCORER_BATCH, shuffle=True, drop_last=True)

    opt_pre = AdamW(binding_scorer.parameters(), lr=2e-4, weight_decay=1e-4)
    sch_pre = CosineAnnealingLR(opt_pre, T_max=SCORER_PRETRAIN_EP, eta_min=1e-6)
    pre_losses = []

    binding_scorer.train()
    for epoch in range(SCORER_PRETRAIN_EP):
        ep_loss = 0.0
        for cdr3_batch, ag_batch in pretrain_loader:
            cdr3_batch = cdr3_batch.to(DEVICE)
            ag_batch   = ag_batch.to(DEVICE)

            z_ab, z_ag = binding_scorer.forward_contrastive(cdr3_batch, ag_batch)
            loss = infonce_loss(z_ab, z_ag, binding_scorer.temperature)

            opt_pre.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(binding_scorer.parameters(), 1.0)
            opt_pre.step()
            ep_loss += loss.item()

        sch_pre.step()
        avg = ep_loss / len(pretrain_loader)
        pre_losses.append(avg)
        if (epoch + 1) % 20 == 0:
            print(f'  Epoch {epoch+1:3d} | InfoNCE loss: {avg:.4f} '
                  f'| temp: {binding_scorer.temperature.item():.3f}')

    torch.save(binding_scorer.state_dict(), SCORER_PRETRAIN_CKPT)
    np.save(CACHE('scorer_pretrain_losses.npy'), np.array(pre_losses))
    print(f'\nSaved pretrained scorer → {SCORER_PRETRAIN_CKPT}')

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(pre_losses, lw=1.2, color='steelblue')
    ax.set(xlabel='Epoch', ylabel='InfoNCE loss', title='Stage 1 contrastive pretraining')
    ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

### 5.3a — Pretraining quality check

Verify that positive pairs score higher than random negative pairs in the
projected 128d space. A well-trained scorer should show clear separation.

In [ ]:
binding_scorer.eval()

# Sample 200 pairs from the SAbDab set for evaluation
eval_n = min(200, len(sabdab_valid))
idx_eval = np.random.choice(len(sabdab_valid), eval_n, replace=False)
cdr3_eval = torch.tensor(sab_cdr3_embs[idx_eval], dtype=torch.float32).to(DEVICE)
ag_eval   = torch.tensor(sab_ag_embs[idx_eval],   dtype=torch.float32).to(DEVICE)

with torch.no_grad():
    z_ab, z_ag = binding_scorer.forward_contrastive(cdr3_eval, ag_eval)
    z_ab_np = z_ab.cpu().numpy()
    z_ag_np = z_ag.cpu().numpy()

# Positive pair similarity (diagonal of cosine sim matrix)
pos_sims = (z_ab_np * z_ag_np).sum(axis=1)

# Negative pair similarity (random shuffled pairs)
shuffled_idx = np.random.permutation(eval_n)
neg_sims = (z_ab_np * z_ag_np[shuffled_idx]).sum(axis=1)

print(f'Positive pair cosine similarity:  mean={pos_sims.mean():.3f}  std={pos_sims.std():.3f}')
print(f'Negative pair cosine similarity:  mean={neg_sims.mean():.3f}  std={neg_sims.std():.3f}')
print(f'Separation (pos - neg mean):  {pos_sims.mean() - neg_sims.mean():.3f}')
print('✓ Good pretraining' if pos_sims.mean() > neg_sims.mean() + 0.1 else
      '⚠ Limited separation — consider more epochs or larger batch')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(pos_sims, bins=30, alpha=0.7, color='steelblue', label='Positive pairs')
ax.hist(neg_sims, bins=30, alpha=0.7, color='coral',     label='Negative pairs')
ax.set(xlabel='Cosine similarity (128d space)', ylabel='Count',
       title='Stage 1: positive vs negative pair similarity')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## Section 5.4 — Stage 2: Fine-tuning on Affinity-Labeled Pairs

**Objective:** Fine-tune `forward_score` (cross-attention mode) to predict
binding affinity (log Kd) from (CDR3 embedding, antigen per-residue embeddings).

**Data sources:**
- **Primary:** SAbDab entries with experimental `affinity_nM` values
- **Clinical mAb subset:** Thera-SAbDab links INN names to SAbDab PDB entries,
  some of which have Kd measurements

**Loss:** MSE on `log10(Kd_nM)` — log-transforming Kd values linearizes the
exponential range of binding affinities and makes the regression well-behaved.

**Label-free fallback:** If no affinity labels are available (all `affinity_nM`
null), the code falls back to a contrastive fine-tuning step that uses the
clinical mAb antigen pairs as additional positive examples. This is weaker but
still improves the scorer's specificity to therapeutic antibody antigens.

In [ ]:
# ── Identify labeled pairs ────────────────────────────────────────────────────
# Primary: SAbDab entries with measured affinities
labeled_mask = sabdab_valid['affinity_nM'].notna()
n_labeled    = labeled_mask.sum()
print(f'SAbDab entries with affinity labels: {n_labeled}')

# Secondary: clinical mAbs in SAbDab (via Thera-SAbDab cross-reference)
# We already have: cdr3_embs (480d) for clinical mAbs
#                  resid_embs (N, ANTIGEN_MAX_LEN, 1280) per-residue antigen embs
# Check if Thera-SAbDab has Kd values for our clinical antibodies
clinical_kd_available = False
if 'Kd' in thera_df.columns or 'affinity' in thera_df.columns:
    kd_col = 'Kd' if 'Kd' in thera_df.columns else 'affinity'
    clinical_kd = thera_df[thera_df[kd_col].notna()][['inn_lower', kd_col]]
    if len(clinical_kd) > 0:
        print(f'Clinical mAbs with Kd in Thera-SAbDab: {len(clinical_kd)}')
        clinical_kd_available = True

print(f'\nFine-tuning strategy: ', end='')
if n_labeled > 0:
    print('MSE regression on log(Kd) labels')
else:
    print('Label-free contrastive fine-tuning (no Kd data available)')

In [ ]:
# ── Prepare per-residue antigen embeddings for labeled SAbDab pairs ───────────
# For cross-attention scoring we need (B, ANTIGEN_MAX_LEN, 1280) padded tensors.
# We compute these on-the-fly for the fine-tuning subset to save memory.

def get_ag_residue_tensor(ag_seq):
    """
    ESM2 per-residue embeddings for a single antigen sequence.
    Returns (ANTIGEN_MAX_LEN, 1280) padded tensor and (ANTIGEN_MAX_LEN,) bool mask.
    True in mask = real position, False = padding.
    """
    _, resid_np = embed_sequence_esm2(ag_seq)  # (L, 1280)
    L = min(len(resid_np), ANTIGEN_MAX_LEN)
    padded = np.zeros((ANTIGEN_MAX_LEN, ESM2_DIM), dtype=np.float32)
    padded[:L] = resid_np[:L]
    mask = np.zeros(ANTIGEN_MAX_LEN, dtype=bool)  # False = real, True = padding
    mask[L:] = True
    return padded, mask

SCORER_FT_CKPT = CACHE('binding_scorer_finetuned.pt')

if os.path.exists(SCORER_FT_CKPT):
    binding_scorer.load_state_dict(
        torch.load(SCORER_FT_CKPT, map_location=DEVICE))
    binding_scorer.eval()
    print(f'Loaded fine-tuned scorer from {SCORER_FT_CKPT}')

elif n_labeled == 0:
    # ── Label-free fallback: contrastive fine-tune on clinical mAb pairs ─────
    print('Label-free fallback: fine-tuning on clinical mAb antigen pairs...')
    # Use clinical mAbs that have both CDR3 emb and valid antigen emb
    clin_valid_idx = np.where(valid_mask)[0]  # indices with valid antigen emb
    print(f'  Clinical mAbs with valid antigen embedding: {len(clin_valid_idx)}')

    if len(clin_valid_idx) >= 8:
        clin_cdr3_t = torch.tensor(cdr3_embs[clin_valid_idx], dtype=torch.float32)
        clin_ag_t   = torch.tensor(mean_embs[clin_valid_idx], dtype=torch.float32)
        ft_ds = TensorDataset(clin_cdr3_t, clin_ag_t)
        ft_loader = DataLoader(ft_ds, batch_size=min(16, len(clin_valid_idx)),
                               shuffle=True, drop_last=True)

        opt_ft = AdamW(binding_scorer.parameters(), lr=5e-5, weight_decay=1e-4)
        binding_scorer.train()
        for epoch in range(SCORER_FINETUNE_EP):
            ep_loss = 0.0
            for cdr3_b, ag_b in ft_loader:
                cdr3_b, ag_b = cdr3_b.to(DEVICE), ag_b.to(DEVICE)
                z_ab, z_ag = binding_scorer.forward_contrastive(cdr3_b, ag_b)
                loss = infonce_loss(z_ab, z_ag, binding_scorer.temperature)
                opt_ft.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(binding_scorer.parameters(), 1.0)
                opt_ft.step(); ep_loss += loss.item()
            if (epoch + 1) % 10 == 0:
                print(f'  Epoch {epoch+1:3d} | contrastive: {ep_loss/len(ft_loader):.4f}')
        torch.save(binding_scorer.state_dict(), SCORER_FT_CKPT)
        print(f'Saved label-free fine-tuned scorer.')
    else:
        print('  Fewer than 8 clinical mAbs with valid antigen embeddings — skipping fine-tune.')
        print('  Scorer stays at Stage 1 pretrained weights.')

else:
    # ── Supervised fine-tuning on log(Kd) labels ─────────────────────────────
    print(f'Stage 2 — supervised fine-tuning on {n_labeled} affinity-labeled pairs')

    labeled_rows = sabdab_valid[labeled_mask].reset_index(drop=True)
    labeled_cdr3 = sab_cdr3_embs[labeled_mask.values]

    # Compute per-residue antigen embs for the labeled subset
    # (expensive but only done for the small labeled subset)
    print('  Computing per-residue antigen embeddings for labeled subset...')
    ft_resid_list, ft_mask_list = [], []
    for i, row in labeled_rows.iterrows():
        padded, mask = get_ag_residue_tensor(row['ag'])
        ft_resid_list.append(padded)
        ft_mask_list.append(mask)
        if (i + 1) % 20 == 0:
            print(f'    {i+1}/{len(labeled_rows)} done')

    ft_resid = np.stack(ft_resid_list)  # (N_labeled, ANTIGEN_MAX_LEN, 1280)
    ft_masks = np.stack(ft_mask_list)   # (N_labeled, ANTIGEN_MAX_LEN) bool

    # Log-transform Kd values; clip to [0.001, 100000] nM for robustness
    kd_vals = labeled_rows['affinity_nM'].clip(0.001, 1e5).values
    log_kd  = np.log10(kd_vals).astype(np.float32)

    ft_ds = TensorDataset(
        torch.tensor(labeled_cdr3,  dtype=torch.float32),
        torch.tensor(ft_resid,      dtype=torch.float32),
        torch.tensor(ft_masks,      dtype=torch.bool),
        torch.tensor(log_kd,        dtype=torch.float32),
    )
    ft_loader = DataLoader(ft_ds, batch_size=16, shuffle=True, drop_last=False)

    # Fine-tune: update all parameters (cross-attn + score_head learn from Kd)
    opt_ft = AdamW(binding_scorer.parameters(), lr=5e-5, weight_decay=1e-4)
    sch_ft = CosineAnnealingLR(opt_ft, T_max=SCORER_FINETUNE_EP, eta_min=1e-6)
    ft_losses = []

    binding_scorer.train()
    for epoch in range(SCORER_FINETUNE_EP):
        ep_loss = 0.0
        for cdr3_b, ag_res_b, ag_mask_b, kd_b in ft_loader:
            cdr3_b    = cdr3_b.to(DEVICE)
            ag_res_b  = ag_res_b.to(DEVICE)
            ag_mask_b = ag_mask_b.to(DEVICE)
            kd_b      = kd_b.to(DEVICE)

            score = binding_scorer.forward_score(cdr3_b, ag_res_b, ag_mask_b)
            # MSE on log(Kd): lower Kd (tighter binding) → higher score
            # We negate: score = -log10(Kd) so higher score means tighter binding
            loss = F.mse_loss(score, -kd_b)

            opt_ft.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(binding_scorer.parameters(), 1.0)
            opt_ft.step(); ep_loss += loss.item()
        sch_ft.step()
        ft_losses.append(ep_loss / len(ft_loader))
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d} | MSE loss: {ft_losses[-1]:.4f}')

    torch.save(binding_scorer.state_dict(), SCORER_FT_CKPT)
    np.save(CACHE('scorer_ft_losses.npy'), np.array(ft_losses))
    print(f'\nSaved fine-tuned scorer → {SCORER_FT_CKPT}')

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(ft_losses, lw=1.2, color='darkorange')
    ax.set(xlabel='Epoch', ylabel='MSE loss (log Kd)', title='Stage 2 affinity fine-tuning')
    ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

binding_scorer.eval()
print('Binding scorer ready.')

---
## Section 6 — Phase 1: Guided ODE Sampling

At each Euler step the binding-scorer gradient is added to the flow model's velocity:

```
v_guided(t, x) = v_flow(t, x) + λ · ∇_x score(inverse_pls(x), z_antigen)
```

**Gradient path:**
```
x  (10d PLS, scaled)
 ↓  inverse_pls_torch — two differentiable linear ops
x_480  (480d AbLang2 CDR3 space, approx)
 ↓  CrossAttentionBindingScorer.forward_score
score  (scalar)
 ↓  torch.autograd.grad
∇_x  (10d)
```

The flow model itself is **frozen** — its weights are never updated.
The gradient only nudges the ODE trajectory at sampling time, not during training.

`GUIDANCE_LAMBDA = 0` reproduces the original unguided pipeline exactly.

In [ ]:
def euler_integrate_guided(
    flow_model,
    binding_scorer,
    x0,                     # (B, PLS_DIM) initial noise
    fw,                     # (B, fw_dim)  framework conditioning
    dev,                    # (B, 2)       developability target
    ag_residues,            # (B, L, 1280) per-residue antigen embs
    ag_pad_mask,            # (B, L) bool, True = padding
    n_steps=N_ODE_STEPS,
    guidance_lambda=GUIDANCE_LAMBDA,
    device=DEVICE,
):
    """
    Custom Euler integrator with per-step binding-scorer gradient guidance.

    At each step:
      1. Compute flow model velocity v_flow (no grad needed)
      2. Compute binding score gradient w.r.t. current x via inverse PLS
      3. Add guidance term: x += dt * (v_flow + λ * ∇_x score)

    Gradient normalisation: ∇_x is clipped to unit norm per sample before
    scaling by λ.  This prevents instability when the scorer produces very
    large gradients early in integration (t ≈ 0, x ≈ noise).
    """
    x = x0.clone()
    dt = 1.0 / n_steps
    t_vals = torch.linspace(0.0, 1.0 - dt, n_steps, device=device)

    flow_model.eval()
    binding_scorer.eval()

    for t_scalar in t_vals:
        t_batch = t_scalar.expand(x.shape[0])

        # ── Flow velocity ──────────────────────────────────────────────────
        with torch.no_grad():
            v_flow = flow_model(t_batch, x, fw, dev)  # (B, 10)

        # ── Binding guidance gradient ──────────────────────────────────────
        if guidance_lambda > 0:
            x_g = x.detach().requires_grad_(True)

            # Differentiable inverse PLS: 10d → 480d
            x_480 = inverse_pls_torch(x_g)   # (B, 480)

            # Binding score via cross-attention
            score = binding_scorer.forward_score(
                x_480, ag_residues, ag_pad_mask)   # (B,)

            # Gradient w.r.t. PLS coordinate
            grad = torch.autograd.grad(
                score.sum(), x_g,
                create_graph=False)[0]              # (B, 10)

            # Safety: replace NaN/Inf
            grad = torch.nan_to_num(grad, nan=0.0, posinf=0.0, neginf=0.0)

            # Normalise: unit L2-norm per sample (keeps λ scale-invariant)
            grad_norm = grad.norm(dim=-1, keepdim=True).clamp(min=1e-8)
            grad_unit = grad / grad_norm           # (B, 10)

            v_total = v_flow + guidance_lambda * grad_unit
        else:
            v_total = v_flow

        x = x.detach() + dt * v_total.detach()

    return x  # (B, PLS_DIM)

In [ ]:
# ── Precompute per-residue antigen embeddings for test antibodies ─────────────
# These are already stored in resid_embs (N, ANTIGEN_MAX_LEN, 1280).
# We convert them to torch tensors once here for fast lookup.

# global_idx → (ANTIGEN_MAX_LEN, 1280) residue emb tensor + (ANTIGEN_MAX_LEN,) mask
ag_resid_tensors = {}
ag_pad_mask_tensors = {}

for global_i in test_idx:
    if valid_mask[global_i]:
        ag_resid_tensors[global_i] = torch.tensor(
            resid_embs[global_i], dtype=torch.float32).to(DEVICE)      # (L, 1280)
        ag_pad_mask_tensors[global_i] = torch.tensor(
            resid_pad_mask[global_i], dtype=torch.bool).to(DEVICE)     # (L,) True=pad

print(f'Test antibodies with per-residue antigen embeddings: '
      f'{sum(valid_mask[i] for i in test_idx)} / {len(test_idx)}')
print(f'Antigen tensor shape: {list(ag_resid_tensors[test_idx[0]].shape)}')

In [ ]:
# ── Identify valid test antibodies ───────────────────────────────────────────
valid_test_local = [
    i for i in range(len(test_idx))
    if isinstance(df['hcdr3_sequence'].values[test_idx[i]], str)
    and len(df['hcdr3_sequence'].values[test_idx[i]]) > 0]
print(f'Valid test antibodies: {len(valid_test_local)}')

# Developability target = 20th percentile of training scores
target_hic  = np.percentile(y_hic_train,  20)
target_sins = np.percentile(y_sins_train, 20)
dev_target_raw = np.array([[target_hic, target_sins]])
dev_target_sc  = dev_scaler.transform(dev_target_raw)
print(f'Dev target: HIC={target_hic:.3f}  SINS={target_sins:.3f}')

In [ ]:
# ── Run Phase 1: guided ODE for all test antibodies ──────────────────────────
flow_targets = {}   # ab_name → (N_FLOW_SAMPLES, PLS_DIM)

print(f'Running Phase 1 (guided ODE, λ={GUIDANCE_LAMBDA}) '
      f'for {len(valid_test_local)} test antibodies...')

for local_i in valid_test_local:
    global_i = test_idx[local_i]
    ab_name  = df.iloc[global_i].get(COL_NAME, f'Ab_{global_i}')

    fw_cond = torch.tensor(X_fw_te[[local_i]],    dtype=torch.float32).to(DEVICE)
    dev_t   = torch.tensor(dev_target_sc,         dtype=torch.float32).to(DEVICE)

    # Expand to N_FLOW_SAMPLES
    fw_exp  = fw_cond.expand(N_FLOW_SAMPLES, -1)   # (50, fw_dim)
    dev_exp = dev_t.expand(N_FLOW_SAMPLES, -1)     # (50, 2)
    x0      = torch.randn(N_FLOW_SAMPLES, PLS_DIM, device=DEVICE)

    has_antigen = global_i in ag_resid_tensors
    if has_antigen and GUIDANCE_LAMBDA > 0:
        ag_res  = ag_resid_tensors[global_i].unsqueeze(0).expand(
            N_FLOW_SAMPLES, -1, -1)                 # (50, L, 1280)
        ag_mask = ag_pad_mask_tensors[global_i].unsqueeze(0).expand(
            N_FLOW_SAMPLES, -1)                     # (50, L)
        lam = GUIDANCE_LAMBDA
    else:
        # No antigen embedding available or guidance disabled — run unguided
        ag_res  = torch.zeros(N_FLOW_SAMPLES, 1, ESM2_DIM, device=DEVICE)
        ag_mask = torch.ones(N_FLOW_SAMPLES, 1, dtype=torch.bool, device=DEVICE)
        lam = 0.0
        if not has_antigen:
            print(f'  ⚠ {ab_name}: no antigen embedding, running unguided')

    targets = euler_integrate_guided(
        flow_model=flow_model,
        binding_scorer=binding_scorer,
        x0=x0,
        fw=fw_exp,
        dev=dev_exp,
        ag_residues=ag_res,
        ag_pad_mask=ag_mask,
        n_steps=N_ODE_STEPS,
        guidance_lambda=lam,
    )
    flow_targets[ab_name] = targets.cpu().numpy()  # (50, 10)

print(f'Phase 1 complete: {len(flow_targets)} × {N_FLOW_SAMPLES} targets')

### 6a — Flow target analysis

In [ ]:
ex_local  = valid_test_local[0]
ex_global = test_idx[ex_local]
ex_name   = df.iloc[ex_global].get(COL_NAME, f'Ab_{ex_global}')
ex_tgts   = flow_targets[ex_name]   # (50, 10)

train_pls = X_pls_tr_sc

nn_dist   = scipy_cdist(ex_tgts, train_pls).min(axis=1).mean()
pw        = scipy_cdist(ex_tgts, ex_tgts)
diversity = pw[np.triu_indices(len(ex_tgts), k=1)].mean()
parent_pt = X_pls_te_sc[[ex_local]]
centroid  = ex_tgts.mean(0, keepdims=True)
shift     = scipy_cdist(parent_pt, centroid)[0, 0]

print(f'Flow target diagnostics — {ex_name}:')
print(f'  NN dist to training manifold: {nn_dist:.3f}')
print(f'  Target diversity:             {diversity:.3f}')
print(f'  Parent → centroid shift:      {shift:.3f}')
print(f'  Guidance λ: {GUIDANCE_LAMBDA}')

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(train_pls[:,0], train_pls[:,1], c='lightgrey', s=25, label='Train')
ax.scatter(ex_tgts[:,0], ex_tgts[:,1], c='steelblue', s=30, alpha=0.6,
           label=f'Flow targets (λ={GUIDANCE_LAMBDA})')
ax.scatter(parent_pt[0,0], parent_pt[0,1], c='red', s=120, marker='*',
           label='Parent CDR3')
ax.set(xlabel='PLS1', ylabel='PLS2',
       title=f'{ex_name}: guided targets vs manifold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

### 6b — Ablation: unguided ODE targets (λ = 0)

Run the same ODE without binding guidance for the first test antibody.
Comparing target centroids shows how much the guidance shifts trajectories.

In [ ]:
ex_fw   = torch.tensor(X_fw_te[[ex_local]], dtype=torch.float32).to(DEVICE)
ex_dev  = torch.tensor(dev_target_sc,       dtype=torch.float32).to(DEVICE)
ex_x0   = torch.randn(N_FLOW_SAMPLES, PLS_DIM, device=DEVICE)

unguided_tgts = euler_integrate_guided(
    flow_model=flow_model,
    binding_scorer=binding_scorer,
    x0=ex_x0,
    fw=ex_fw.expand(N_FLOW_SAMPLES, -1),
    dev=ex_dev.expand(N_FLOW_SAMPLES, -1),
    ag_residues=torch.zeros(N_FLOW_SAMPLES, 1, ESM2_DIM, device=DEVICE),
    ag_pad_mask=torch.ones(N_FLOW_SAMPLES, 1, dtype=torch.bool, device=DEVICE),
    n_steps=N_ODE_STEPS,
    guidance_lambda=0.0,
).cpu().numpy()

guided_center   = ex_tgts.mean(0)
unguided_center = unguided_tgts.mean(0)
shift_mag = np.linalg.norm(guided_center - unguided_center)
print(f'Centroid shift (guided vs unguided): {shift_mag:.4f} PLS units')

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(train_pls[:,0], train_pls[:,1], c='lightgrey', s=20, label='Train')
ax.scatter(unguided_tgts[:,0], unguided_tgts[:,1], c='coral', s=25, alpha=0.5,
           label='Unguided (λ=0)')
ax.scatter(ex_tgts[:,0], ex_tgts[:,1], c='steelblue', s=25, alpha=0.5,
           label=f'Guided (λ={GUIDANCE_LAMBDA})')
ax.scatter(*parent_pt[0], c='red', s=120, marker='*', label='Parent')
ax.arrow(*unguided_center[:2], *(guided_center[:2] - unguided_center[:2]),
         head_width=0.05, color='black', length_includes_head=True)
ax.set(xlabel='PLS1', ylabel='PLS2', title='Guided vs unguided ODE centroid shift')
ax.legend(fontsize=8); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

---
## Section 7 — Phase 2: AbLang2 Masked Pass + Steered Sampling

Phase 2 is **unchanged** from the original pipeline.  The only difference is that
the `flow_targets` it uses now come from the binding-guided ODE of Phase 1.

In [ ]:
def get_cdr3_mlm_logits(ablang, masked_vh, vl_seq, cdr3_positions):
    with torch.no_grad():
        reps = ablang([(masked_vh, vl_seq)], mode='rescoding')
    hidden_np = np.array(reps[0])
    vh_len    = len(masked_vh)
    vl_len    = len(vl_seq)
    n_ex      = hidden_np.shape[0] - vh_len - vl_len
    vh_hid    = hidden_np[:vh_len]
    cdr3_hid  = vh_hid[cdr3_positions]
    cdr3_t    = torch.tensor(cdr3_hid, dtype=torch.float32).to(DEVICE)
    try:
        with torch.no_grad(): logits = ablang.AbLang.AbHead(cdr3_t)
    except AttributeError:
        for attr in ['AbHead', 'lm_head', 'predictions', 'cls']:
            head = getattr(ablang.AbLang, attr, None)
            if head is not None:
                with torch.no_grad(): logits = head(cdr3_t); break
        else:
            raise RuntimeError('Cannot find AbLang2 MLM head.')
    try:
        vocab      = ablang.tokenizer.vocab
        aa_indices = torch.tensor(
            [vocab[aa] for aa in AA_VOCAB if aa in vocab], dtype=torch.long)
        aa_logits  = logits[:, aa_indices] if len(aa_indices) == 20 else logits[:, :20]
    except (AttributeError, KeyError):
        aa_logits = logits[:, :20]
    return F.log_softmax(aa_logits, dim=-1).cpu().numpy()


def temperature_sample_sequences(log_probs, M=20, temperature=1.0, seed=None):
    if seed is not None: np.random.seed(seed)
    probs = np.exp(log_probs / temperature)
    probs = probs / probs.sum(axis=1, keepdims=True)
    L = log_probs.shape[0]
    return [''.join(AA_VOCAB[np.random.choice(20, p=probs[pos])] for pos in range(L))
            for _ in range(M)]


def splice_cdr3(parent_vh, cdr3_positions, new_cdr3):
    chars = list(parent_vh)
    for j, pos in enumerate(cdr3_positions):
        if pos < len(chars) and j < len(new_cdr3): chars[pos] = new_cdr3[j]
    return ''.join(chars)


def rescore_sequences(ablang, spliced_vhs, parent_vl, cdr3_positions):
    seqs = [(vh, parent_vl) for vh in spliced_vhs]
    with torch.no_grad():
        reps = ablang(seqs, mode='rescoding')
    embs = []
    for j, vh in enumerate(spliced_vhs):
        hidden = np.array(reps[j])
        h_hid  = hidden[:len(vh)]
        valid  = [k for k in cdr3_positions if k < len(h_hid)]
        embs.append(h_hid[valid].mean(0) if valid else np.zeros(480))
    return np.vstack(embs)


def guided_sample_sequences(log_probs, flow_target_pls, parent_vh, parent_vl,
                             cdr3_idx, ablang, pls_model, pls_scaler,
                             M=20, temperature=1.0, guidance_strength=2.0,
                             top_k_tokens=5):
    L      = log_probs.shape[0]
    greedy = [AA_VOCAB[log_probs[pos].argmax()] for pos in range(L)]
    steered_log_probs = log_probs.copy()

    for pos in range(L):
        base_probs = np.exp(log_probs[pos] / temperature)
        base_probs /= base_probs.sum()
        top_k_idx   = np.argsort(base_probs)[-top_k_tokens:]
        steering_bonus = np.zeros(20)

        for aa_idx in top_k_idx:
            trial = greedy.copy(); trial[pos] = AA_VOCAB[aa_idx]
            trial_vh = splice_cdr3(parent_vh, cdr3_idx, ''.join(trial))
            with torch.no_grad():
                reps = ablang([(trial_vh, parent_vl)], mode='rescoding')
            hidden   = np.array(reps[0])[:len(trial_vh)]
            valid    = [i for i in cdr3_idx if i < len(hidden)]
            cdr3_emb = hidden[valid].mean(0, keepdims=True)
            cdr3_pls = pls_scaler.transform(pls_model.transform(cdr3_emb))
            dist     = np.linalg.norm(cdr3_pls - flow_target_pls)
            steering_bonus[aa_idx] = -dist

        steering_bonus -= steering_bonus[top_k_idx].max()
        steering_weight = np.exp(guidance_strength * steering_bonus)
        steering_weight[steering_bonus == 0] = 0
        combined  = base_probs * (1 + steering_weight)
        combined /= combined.sum()
        steered_log_probs[pos] = np.log(combined + 1e-12)

    return temperature_sample_sequences(steered_log_probs, M=M,
                                        temperature=temperature)

In [ ]:
GUIDANCE_STRENGTH = 2.0
TOP_K_TOKENS      = 5
generation_records = []

print(f'Running Phase 2 (steered sequence generation) '
      f'for {len(valid_test_local)} test antibodies...')

for local_i in valid_test_local:
    global_i    = test_idx[local_i]
    row         = df.iloc[global_i]
    ab_name     = row.get(COL_NAME, f'Ab_{global_i}')
    parent_vh   = row[COL_VH]
    parent_vl   = row[COL_VL]
    cdr3_idx    = row['h_cdr3_idx']
    parent_cdr3 = row['hcdr3_sequence']

    masked_vh  = mask_cdr3(parent_vh, cdr3_idx)
    log_probs  = get_cdr3_mlm_logits(ablang, masked_vh, parent_vl, cdr3_idx)

    flow_centroid = flow_targets[ab_name].mean(0, keepdims=True)

    cdr3_candidates = guided_sample_sequences(
        log_probs=log_probs, flow_target_pls=flow_centroid,
        parent_vh=parent_vh, parent_vl=parent_vl,
        cdr3_idx=cdr3_idx, ablang=ablang,
        pls_model=pls, pls_scaler=pls_scaler,
        M=M_SEQUENCES, temperature=TEMPERATURE,
        guidance_strength=GUIDANCE_STRENGTH,
        top_k_tokens=TOP_K_TOKENS)

    spliced_vhs = [splice_cdr3(parent_vh, cdr3_idx, c) for c in cdr3_candidates]

    generation_records.append({
        'ab_name': ab_name, 'global_idx': global_i, 'local_idx': local_i,
        'parent_vh': parent_vh, 'parent_vl': parent_vl,
        'parent_cdr3': parent_cdr3, 'cdr3_idx': cdr3_idx,
        'cdr3_candidates': cdr3_candidates, 'spliced_vhs': spliced_vhs,
    })
    print(f'  ✓ {ab_name}  len={len(parent_cdr3)}')

print(f'\nPhase 2 complete: {len(generation_records)} antibodies processed.')

---
## Section 8 — Phase 3: Oracle Scoring

In [ ]:
def oracle_score(vh_seq, vl_seq):
    with torch.no_grad():
        emb = ablang([(vh_seq, vl_seq)], mode='seqcoding')
    emb_sc = full_scaler.transform(np.array(emb).reshape(1, -1))
    return {
        'hic':  float(oracle_hic.predict(emb_sc)[0]),
        'sins': float(oracle_sins.predict(emb_sc)[0]),
        'full_emb_sc': emb_sc[0],
    }

results = []

for rec in generation_records:
    ab_name = rec['ab_name']
    par = oracle_score(rec['parent_vh'], rec['parent_vl'])

    best_score, best = None, None
    for sj, (cdr3, vh) in enumerate(
            zip(rec['cdr3_candidates'], rec['spliced_vhs'])):
        sc = oracle_score(vh, rec['parent_vl'])
        zh = (sc['hic']  - y_hic_train.mean()) / (y_hic_train.std()  + 1e-8)
        zs = (sc['sins'] - y_sins_train.mean()) / (y_sins_train.std() + 1e-8)
        if best_score is None or (zh + zs) < best_score:
            best_score = zh + zs; best = (sj, sc)

    ung = oracle_score(rec['spliced_vhs'][0], rec['parent_vl'])

    results.append({
        'ab_name':              ab_name,
        'parent_vh':            rec['parent_vh'],
        'parent_vl':            rec['parent_vl'],
        'parent_cdr3':          rec['parent_cdr3'],
        'guided_cdr3':          rec['cdr3_candidates'][best[0]],
        'guided_vh':            rec['spliced_vhs'][best[0]],
        'unguided_cdr3':        rec['cdr3_candidates'][0],
        'unguided_vh':          rec['spliced_vhs'][0],
        'oracle_hic_parent':    par['hic'],
        'oracle_sins_parent':   par['sins'],
        'oracle_hic_guided':    best[1]['hic'],
        'oracle_sins_guided':   best[1]['sins'],
        'oracle_hic_unguided':  ung['hic'],
        'oracle_sins_unguided': ung['sins'],
        'h_cdr3_idx':           rec['cdr3_idx'],
    })
    print(f'  ✓ {ab_name}  '
          f'HIC: {par["hic"]:+.3f}→{best[1]["hic"]:+.3f}  '
          f'SINS: {par["sins"]:+.3f}→{best[1]["sins"]:+.3f}')

seq_df = pd.DataFrame(results)
seq_df.to_csv(CACHE('seq_df.csv'), index=False)
print(f'\nSaved {len(seq_df)} results.')

---
## Section 9 — Baselines

Three comparisons:
1. **AbLang2-only** (no flow model, no binding guidance)
2. **Flow-only** (flow model ODE with λ=0, no binding guidance)
3. **Flow + binding guidance** (our full pipeline, λ > 0)

In [ ]:
# ── Baseline A: AbLang2-only ──────────────────────────────────────────────────
baseline_records = []
print('Baseline A — AbLang2-only (no flow, no binding guidance)...')

for local_i in valid_test_local:
    global_i  = test_idx[local_i]
    row       = df.iloc[global_i]
    ab_name   = row.get(COL_NAME, f'Ab_{global_i}')

    log_probs  = get_cdr3_mlm_logits(
        ablang, mask_cdr3(row[COL_VH], row['h_cdr3_idx']),
        row[COL_VL], row['h_cdr3_idx'])
    candidates = temperature_sample_sequences(
        log_probs, M=M_SEQUENCES, temperature=TEMPERATURE, seed=int(global_i))
    spliced    = [splice_cdr3(row[COL_VH], row['h_cdr3_idx'], c) for c in candidates]

    best_score, best_hic, best_sins, best_cdr3 = None, None, None, None
    for cdr3, vh in zip(candidates, spliced):
        sc = oracle_score(vh, row[COL_VL])
        zh = (sc['hic']  - y_hic_train.mean()) / (y_hic_train.std()  + 1e-8)
        zs = (sc['sins'] - y_sins_train.mean()) / (y_sins_train.std() + 1e-8)
        if best_score is None or (zh + zs) < best_score:
            best_score = zh + zs
            best_hic, best_sins, best_cdr3 = sc['hic'], sc['sins'], cdr3

    baseline_records.append({
        'ab_name': ab_name, 'baseline_cdr3': best_cdr3,
        'oracle_hic_baseline': best_hic, 'oracle_sins_baseline': best_sins,
    })
    print(f'  ✓ {ab_name}  HIC={best_hic:.3f}  SINS={best_sins:.3f}')

baseline_df = pd.DataFrame(baseline_records)

In [ ]:
# ── Baseline B: flow model only (λ = 0) ─────────────────────────────────────
print('Baseline B — flow model + Phase 2, no binding guidance (λ=0)...')
flow_only_records = []

for local_i in valid_test_local:
    global_i  = test_idx[local_i]
    row       = df.iloc[global_i]
    ab_name   = row.get(COL_NAME, f'Ab_{global_i}')

    x0  = torch.randn(N_FLOW_SAMPLES, PLS_DIM, device=DEVICE)
    fw_ = torch.tensor(X_fw_te[[local_i]], dtype=torch.float32).to(DEVICE)
    dev_= torch.tensor(dev_target_sc, dtype=torch.float32).to(DEVICE)

    tgts_unguided = euler_integrate_guided(
        flow_model=flow_model, binding_scorer=binding_scorer,
        x0=x0, fw=fw_.expand(N_FLOW_SAMPLES,-1), dev=dev_.expand(N_FLOW_SAMPLES,-1),
        ag_residues=torch.zeros(N_FLOW_SAMPLES,1,ESM2_DIM,device=DEVICE),
        ag_pad_mask=torch.ones(N_FLOW_SAMPLES,1,dtype=torch.bool,device=DEVICE),
        n_steps=N_ODE_STEPS, guidance_lambda=0.0
    ).cpu().numpy()

    centroid_ung = tgts_unguided.mean(0, keepdims=True)
    log_probs    = get_cdr3_mlm_logits(
        ablang, mask_cdr3(row[COL_VH], row['h_cdr3_idx']),
        row[COL_VL], row['h_cdr3_idx'])
    candidates   = guided_sample_sequences(
        log_probs=log_probs, flow_target_pls=centroid_ung,
        parent_vh=row[COL_VH], parent_vl=row[COL_VL],
        cdr3_idx=row['h_cdr3_idx'], ablang=ablang,
        pls_model=pls, pls_scaler=pls_scaler,
        M=M_SEQUENCES, temperature=TEMPERATURE,
        guidance_strength=GUIDANCE_STRENGTH, top_k_tokens=TOP_K_TOKENS)
    spliced = [splice_cdr3(row[COL_VH], row['h_cdr3_idx'], c) for c in candidates]

    best_score, best_hic, best_sins, best_cdr3 = None, None, None, None
    for cdr3, vh in zip(candidates, spliced):
        sc = oracle_score(vh, row[COL_VL])
        zh = (sc['hic']  - y_hic_train.mean()) / (y_hic_train.std()  + 1e-8)
        zs = (sc['sins'] - y_sins_train.mean()) / (y_sins_train.std() + 1e-8)
        if best_score is None or (zh + zs) < best_score:
            best_score = zh + zs
            best_hic, best_sins, best_cdr3 = sc['hic'], sc['sins'], cdr3

    flow_only_records.append({
        'ab_name': ab_name, 'flow_only_cdr3': best_cdr3,
        'oracle_hic_flow_only': best_hic, 'oracle_sins_flow_only': best_sins,
    })

flow_only_df = pd.DataFrame(flow_only_records)

# ── Merge and summarise ───────────────────────────────────────────────────────
compare_df = seq_df.merge(
    baseline_df[['ab_name','oracle_hic_baseline','oracle_sins_baseline','baseline_cdr3']],
    on='ab_name', how='inner')
compare_df = compare_df.merge(
    flow_only_df[['ab_name','oracle_hic_flow_only','oracle_sins_flow_only']],
    on='ab_name', how='inner')

print(f'\n{"Condition":<30} {"HIC":>14} {"SINS":>14}')
print('─' * 60)
for label, hc, sc_ in [
        ('Parent',                    'oracle_hic_parent',   'oracle_sins_parent'),
        ('AbLang2 only',              'oracle_hic_baseline', 'oracle_sins_baseline'),
        ('Flow only  (λ=0)',          'oracle_hic_flow_only','oracle_sins_flow_only'),
        (f'Flow + binding (λ={GUIDANCE_LAMBDA})', 'oracle_hic_guided', 'oracle_sins_guided')]:
    h = compare_df[hc]; s = compare_df[sc_]
    print(f'{label:<30} {h.mean():>+6.3f}±{h.sem():.3f}  {s.mean():>+6.3f}±{s.sem():.3f}')

---
## Section 10 — Save Artifacts for Evaluation Notebook

In [ ]:
import json

compare_df.to_csv(f'{EVAL_DIR}/compare_df.csv', index=False)
seq_df.to_csv(    f'{EVAL_DIR}/seq_df.csv',     index=False)
flow_only_df.to_csv(f'{EVAL_DIR}/flow_only_df.csv', index=False)

df[['antibody_name','vh_protein_sequence','vl_protein_sequence',
    'hcdr3_sequence','hcdr3_len','hierarchical_cluster_fold',
    'liability_count','HIC','AC-SINS_pH7.4']].to_csv(
    f'{EVAL_DIR}/df_sequences.csv', index=False)

cdr3_idx_dict = {int(i): df['h_cdr3_idx'].iloc[i]
                 for i in range(len(df))
                 if isinstance(df['h_cdr3_idx'].iloc[i], list)}
with open(f'{EVAL_DIR}/cdr3_idx.json', 'w') as f:
    json.dump(cdr3_idx_dict, f)

# Oracle models + scalers
joblib.dump(oracle_hic,   f'{EVAL_DIR}/oracle_hic.pkl')
joblib.dump(oracle_sins,  f'{EVAL_DIR}/oracle_sins.pkl')
joblib.dump(full_scaler,  f'{EVAL_DIR}/full_scaler.pkl')
joblib.dump(cdr3_scaler,  f'{EVAL_DIR}/cdr3_scaler.pkl')
joblib.dump(fw_scaler,    f'{EVAL_DIR}/fw_scaler.pkl')
joblib.dump(pls,          f'{EVAL_DIR}/pls_model.pkl')
joblib.dump(pls_scaler,   f'{EVAL_DIR}/pls_scaler.pkl')
joblib.dump(dev_scaler,   f'{EVAL_DIR}/dev_scaler.pkl')

# PCA objects
joblib.dump(pca, f'{EVAL_DIR}/pca_cdr3.pkl')
full_pca = PCA(n_components=20, random_state=42).fit(X_full_tr)
joblib.dump(full_pca, f'{EVAL_DIR}/pca_full.pkl')

# Binding scorer (fine-tuned checkpoint)
torch.save(binding_scorer.state_dict(), f'{EVAL_DIR}/binding_scorer.pt')

# Arrays
np.save(f'{EVAL_DIR}/y_hic_train.npy',  y_hic_train)
np.save(f'{EVAL_DIR}/y_sins_train.npy', y_sins_train)
np.save(f'{EVAL_DIR}/y_hic_test.npy',   y_hic_test)
np.save(f'{EVAL_DIR}/y_sins_test.npy',  y_sins_test)
np.save(f'{EVAL_DIR}/train_idx.npy', train_idx)
np.save(f'{EVAL_DIR}/test_idx.npy',  test_idx)
np.save(f'{EVAL_DIR}/X_cdr3_tr.npy', X_cdr3_tr)
np.save(f'{EVAL_DIR}/X_cdr3_te.npy', X_cdr3_te)
np.save(f'{EVAL_DIR}/X_fw_te.npy',   X_fw_te)
np.save(f'{EVAL_DIR}/X_full_tr.npy', X_full_tr)
np.save(f'{EVAL_DIR}/X_full_te.npy', X_full_te)

# PLS inverse tensors (for eval notebook guidance)
np.save(f'{EVAL_DIR}/pls_loadings_T.npy', pls.x_loadings_.T.astype(np.float32))
np.save(f'{EVAL_DIR}/pls_mean.npy',       pls.x_mean_.astype(np.float32))

print(f'All artifacts saved to: {EVAL_DIR}')
for fname in sorted(os.listdir(EVAL_DIR)):
    size = os.path.getsize(f'{EVAL_DIR}/{fname}') / 1024
    print(f'  {fname:<42} {size:>8.1f} KB')